In [2]:
import pandas as pd

In [3]:
df.head()

NameError: name 'df' is not defined

In [4]:
df = pd.read_csv("./datasets/data.csv")

In [5]:
# Delete html and urls
df['Sentence'] = df['Sentence'].str.replace(r'<[^>]+>', '', regex=True)
df['Sentence'] = df['Sentence'].str.replace(r'http\S+|www\S+|https\S+', '', regex=True)

# Remove special characters and digits
df['Sentence'] = df['Sentence'].str.replace(r'[^a-zA-Z\s]', '', regex=True)

# Convert to lowercase
df['Sentence'] = df['Sentence'].str.lower()

# NER and erplace by <ORG>

import spacy
nlp = spacy.load("en_core_web_sm")
def replace_org_entities(text):
    doc = nlp(text)
    for ent in doc.ents:
        if ent.label_ == "ORG":
            text = text.replace(ent.text, "<ORG>")
    return text

df['Sentence'] = df['Sentence'].apply(replace_org_entities)

#stopwords
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
df['Sentence'] = df['Sentence'].apply(lambda x: ' '.join([word for word in x.split() if word not in stop_words]))

df["Sentiment"] = df["Sentiment"].apply(lambda x: 1 if x == "positive" else -1 if x == "negative" else 0)

# Save the cleaned data
df.to_csv("./datasets/cleaned_data.csv", index=False)

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/ayalamanu/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [29]:
corpus = df['Sentence'].tolist()

In [30]:
from collections import Counter
from itertools import islice
import re
word_counts = Counter()
for doc in corpus:
    # extrae solo secuencias de caracteres alfanuméricos
    words = re.findall(r"\b\w+\b", doc.lower())
    word_counts.update(words)

# 1.2 Construye un contador de bigramas (opcional)
bigram_counts = Counter()
for doc in corpus:
    words = re.findall(r"\b\w+\b", doc.lower())
    bigram_counts.update(zip(words, words[1:]))

candidates = []

vocab = tokenizer.get_vocab()  # dict token → idx

# 2.1 candidatos unigramas
uni_candidates = [
    tok for tok, freq in word_counts.items()
    if freq > 100    # umbral de frecuencia
    and tok not in vocab
]

# 2.2 candidatos bigramas (concatenados con guión o '_')
bi_candidates = [
    f"{w1}_{w2}" for (w1, w2), freq in bigram_counts.items()
    if freq > 50
    and f"{w1}_{w2}" not in vocab
]
print("Top 20 unigrams candidatos:", list(islice(uni_candidates, 20)))
print("Top 20 bigrams candidatos:", list(islice(bi_candidates, 20)))

Top 20 unigrams candidatos: ['eur']
Top 20 bigrams candidatos: ['eur_mn']


In [15]:
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = "ProsusAI/finbert"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
model.eval()

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

In [ ]:
def embed_text(text):
    all_embeddings = []
    for i in range(0, len(text), 32):  # batch size de 32
        batch = text[i:i+32]
        enc = tokenizer(batch, padding=True, truncation=True, return_tensors='pt', max_length=512)
        with torch.no_grad():
            outputs = model(**enc)
        
        all_embeddings.append(outputs.last_hidden_state.mean(dim=1).cpu())
    
    return all_embeddings

# Embed the corpus
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

corpus_embeddings = embed_text(corpus)
import numpy as np
corpus_embeddings = np.vstack(corpus_embeddings)
# Save the embeddings
np.save("./datasets/corpus_embeddings.npy", corpus_embeddings)
